# Práctica Obligatoria: Transfer Learning y Fine Tuning con CNN

## 1. Importar librerías

In [23]:
import os
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

## 2. Preparación del dataset

In [24]:
# Cambia esta ruta según dónde tengas el dataset de paisajes
DATASET_DIR = "data"

TRAIN_DIRS = [
    os.path.join(DATASET_DIR, "github_train_0"),
    os.path.join(DATASET_DIR, "github_train_1"),
    os.path.join(DATASET_DIR, "github_train_2"),
    os.path.join(DATASET_DIR, "github_train_3")
]

TEST_DIR = os.path.join(DATASET_DIR, "github_test")

IMG_SIZE = (224,224)
BATCH_SIZE = 32

### Generadores de datos

In [25]:
import pandas as pd

def build_dataframe(directory):

    files = []
    labels = []

    for file in os.listdir(directory):

        filepath = os.path.join(directory, file)

        if "cat" in file.lower():
            label = "cat"
        elif "dog" in file.lower():
            label = "dog"
        else:
            continue

        files.append(filepath)
        labels.append(label)

    return pd.DataFrame({
        "filename": files,
        "class": labels
    })


train_df = pd.concat([
    build_dataframe("data/github_train_0"),
    build_dataframe("data/github_train_1"),
    build_dataframe("data/github_train_2"),
    build_dataframe("data/github_train_3")
])

test_df = build_dataframe("data/github_test")

In [26]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [27]:
val_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filename",
    y_col="class",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

Found 800 validated image filenames belonging to 2 classes.


In [28]:
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filename",
    y_col="class",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 1000 validated image filenames belonging to 2 classes.


### Función para visualizar imágenes

In [29]:
def show_images_from_generator(generator, n=9):
    """
    Muestra n imágenes de un generador (train/val/test) de ImageDataGenerator.
    """
    X_batch, y_batch = next(generator)  # toma un batch del generador
    plt.figure(figsize=(6,6))
    for i in range(n):
        plt.subplot(3,3,i+1)
        plt.imshow((X_batch[i] + 1)/2)  # MobileNetV2 preprocesa a [-1,1], por eso lo normalizamos para mostrar
        plt.title(generator.class_indices.keys()[np.argmax(y_batch[i])])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

In [30]:
show_images_from_generator(train_generator, n=9)

NameError: name 'train_generator' is not defined

## 3. Modelo preentrenado: MobileNetV2

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

base_model.summary()

## 4. Transfer Learning

In [ ]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dense(64, activation='relu')(x)

outputs = layers.Dense(train_generator.num_classes, activation='softmax')(x)

model = keras.Model(inputs=base_model.input, outputs=outputs)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

### Entrenamiento Transfer Learning

In [ ]:
history_tl = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

### Evaluación en test

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print("Test accuracy:", test_acc)

In [ ]:
preds = model.predict(test_generator)
y_pred = np.argmax(preds, axis=1)
y_true = test_generator.classes

print(classification_report(y_true, y_pred, target_names=test_generator.class_indices.keys()))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=test_generator.class_indices.keys(),
            yticklabels=test_generator.class_indices.keys())
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Transfer Learning")
plt.show()

## 5. Fine Tuning

In [ ]:
# Descongelar las últimas capas
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_ft = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

### Evaluación tras Fine Tuning

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print("Test accuracy after fine tuning:", test_acc)

In [ ]:
preds = model.predict(test_generator)
y_pred = np.argmax(preds, axis=1)

print(classification_report(y_true, y_pred, target_names=test_generator.class_indices.keys()))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=test_generator.class_indices.keys(),
            yticklabels=test_generator.class_indices.keys())

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Fine Tuning")
plt.show()

## 6. Comparación de resultados

En esta sección se deben comparar los resultados obtenidos con:

- CNN creada manualmente en la práctica anterior
- Modelo con **Transfer Learning**
- Modelo con **Fine Tuning**

Normalmente:

- Transfer Learning mejora bastante respecto a CNN pequeñas.
- Fine Tuning suele mejorar aún más al ajustar capas profundas.
- También aumenta el coste computacional.
